<a href="https://colab.research.google.com/github/Chosencodes/Anatomy-Prior-Guided-Multi-Organ-Segmentation/blob/main/Anatomy_Prior_Guided_Multi_Organ_CT_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib as plt
import os
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
root = Path("/content/drive/MyDrive/anatomy-prior-segmentation/data")

organs = [
    "liver",
    "spleen",
    "kidney_left",
    "kidney_right",
    "pancreas",
    "stomach",
    "urinary_bladder"
]

patient_id = "s0245"
ct_path = root / patient_id / "ct.nii.gz"
ct_img = nib.load(ct_path)
ct_data = ct_img.get_fdata()

print("CT shape:", ct_data.shape)
print("Voxel spacing:", ct_img.header.get_zooms())

masks = {}
for organ in organs:
    mask_path = root / patient_id / "segmentations" / f"{organ}.nii.gz"
    mask_img = nib.load(mask_path)
    masks[organ] = mask_img.get_fdata()
    print(f"{organ}: {int(np.sum(masks[organ] > 0))} voxels")

In [ ]:
mid_slice = ct_data.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(ct_data[:, :, mid_slice].T, cmap="gray", origin="lower")
axes[0].set_title(f"{patient_id} - CT slice {mid_slice}")
axes[0].axis("off")

axes[1].imshow(ct_data[:, :, mid_slice].T, cmap="gray", origin="lower")

colors = plt.cm.tab10(np.linspace(0, 1, len(organs)))
for organ, color in zip(organs, colors):
    mask_slice = masks[organ][:, :, mid_slice]
    if mask_slice.sum() > 0:
        overlay = np.ma.masked_where(mask_slice == 0, mask_slice)
        axes[1].imshow(overlay.T, cmap=plt.cm.colors.ListedColormap([color]), alpha=0.5, origin="lower")

axes[1].set_title(f"{patient_id} - Organs overlaid")
axes[1].axis("off")

plt.tight_layout()
plt.show()